In [1]:
import os
import json
import numpy as np

# =========================================================
# Configuration
# =========================================================

SEED = 42
TRIALS_PER_N = 5000
N_VALUES = list(range(3, 10))  # n = 3,...,9

OUTPUT_DIR = "discrete_shared_matrices"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MATRIX_BANK_PATH = os.path.join(
    OUTPUT_DIR,
    f"ordinal_matrix_bank_seed_{SEED}_trials_{TRIALS_PER_N}.npz"
)

METADATA_PATH = os.path.join(
    OUTPUT_DIR,
    f"ordinal_matrix_bank_seed_{SEED}_trials_{TRIALS_PER_N}_metadata.json"
)


# =========================================================
# Generate one ordinal reciprocal matrix
# =========================================================

def generate_one_ordinal_pcm(n, rng):
    """
    Generate one ordinal reciprocal PCM encoded as exponents:

        1  = strict preference P
        0  = tie
       -1  = reciprocal strict preference 1/P

    Later, for a given alpha:

        A(alpha) = alpha ** O
    """

    O = np.zeros((n, n), dtype=np.int8)

    for i in range(n):
        for j in range(i + 1, n):

            value = rng.choice([-1, 0, 1])

            O[i, j] = value
            O[j, i] = -value

    return O


# =========================================================
# Generate and save matrix bank
# =========================================================

rng = np.random.default_rng(SEED)

matrix_bank = {}

for n in N_VALUES:
    matrices_n = np.zeros((TRIALS_PER_N, n, n), dtype=np.int8)

    for matrix_id in range(TRIALS_PER_N):
        matrices_n[matrix_id] = generate_one_ordinal_pcm(n, rng)

    matrix_bank[f"n_{n}"] = matrices_n

    print(f"Generated {TRIALS_PER_N} matrices for n={n}")

np.savez_compressed(MATRIX_BANK_PATH, **matrix_bank)

metadata = {
    "seed": SEED,
    "trials_per_n": TRIALS_PER_N,
    "n_values": N_VALUES,
    "encoding": {
        "1": "strict preference P",
        "0": "tie",
        "-1": "reciprocal strict preference 1/P"
    },
    "cardinalisation": "A(alpha) = alpha ** O",
    "description": (
        "Ordinal reciprocal matrices generated once and reused for all "
        "priority derivation methods."
    )
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print("\nMatrix bank saved to:")
print(MATRIX_BANK_PATH)

print("\nMetadata saved to:")
print(METADATA_PATH)

Generated 5000 matrices for n=3
Generated 5000 matrices for n=4
Generated 5000 matrices for n=5
Generated 5000 matrices for n=6
Generated 5000 matrices for n=7
Generated 5000 matrices for n=8
Generated 5000 matrices for n=9

Matrix bank saved to:
discrete_shared_matrices\ordinal_matrix_bank_seed_42_trials_5000.npz

Metadata saved to:
discrete_shared_matrices\ordinal_matrix_bank_seed_42_trials_5000_metadata.json


In [ ]:
#Experimento 1
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.linalg import eig

# =========================================================
# Configuration
# =========================================================

SEED = 42
TRIALS_PER_N = 5000

# Reviewer requested clarification about n = 3.
N_VALUES = list(range(3, 10))  # n = 3, ..., 9

# Discrete Saaty-scale strict intensities.
# Ties are always equal to 1.
ALPHA_VALUES = list(range(2, 10))  # alpha = 2, ..., 9
BASELINE_ALPHA = 2

# Explicit ranking tolerance requested by reviewers.
RANK_TOL = 1e-10

# Folder where the saved matrix bank is located.
INPUT_DIR = "discrete_shared_matrices"

MATRIX_BANK_PATH = os.path.join(
    INPUT_DIR,
    f"ordinal_matrix_bank_seed_{SEED}_trials_{TRIALS_PER_N}.npz"
)

# Output folder for this experiment.
OUTPUT_DIR = "discrete_evm_reviewer_ready"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Saaty's random index values.
# These are the standard values commonly used in AHP.
RI_VALUES = {
    1: 0.00,
    2: 0.00,
    3: 0.58,
    4: 0.90,
    5: 1.12,
    6: 1.24,
    7: 1.32,
    8: 1.41,
    9: 1.45,
    10: 1.49,
}


# =========================================================
# Load saved ordinal matrices
# =========================================================

def load_matrix_bank():
    """
    Load the ordinal matrices generated previously.

    The stored matrices O are encoded as:
        O[i,j] =  1  means strict preference P,
        O[i,j] =  0  means tie,
        O[i,j] = -1  means reciprocal strict preference 1/P.

    For a given intensity alpha, the cardinal PCM is:
        A(alpha) = alpha ** O.
    """

    if not os.path.exists(MATRIX_BANK_PATH):
        raise FileNotFoundError(
            f"Matrix bank not found:\n{MATRIX_BANK_PATH}\n\n"
            "Run the matrix-generation script first."
        )

    data = np.load(MATRIX_BANK_PATH)

    matrix_bank = {}

    for key in data.files:
        n = int(key.split("_")[1])
        matrix_bank[n] = data[key]

    return matrix_bank


def substitute_alpha(O, alpha):
    """
    Convert an ordinal exponent matrix O into a cardinal reciprocal PCM.

    If O[i,j] = 1, then A[i,j] = alpha.
    If O[i,j] = 0, then A[i,j] = 1.
    If O[i,j] = -1, then A[i,j] = 1/alpha.
    """

    return alpha ** O.astype(float)


# =========================================================
# Eigenvector method
# =========================================================

def principal_eigenpair(A):
    """
    Compute the Perron eigenvalue and the associated priority vector.
    """

    eigenvalues, eigenvectors = eig(A)

    idx = np.argmax(eigenvalues.real)

    lambda_max = eigenvalues[idx].real
    w = eigenvectors[:, idx].real

    # Eigenvectors have arbitrary sign.
    if np.sum(w) < 0:
        w = -w

    # Numerical safeguard. For positive reciprocal matrices, the Perron vector
    # should be positive, but small numerical artefacts may appear.
    if np.any(w <= 0):
        w = np.abs(w)

    w = w / np.sum(w)

    return lambda_max, w


def eigenvector_priority(A):
    """
    Eigenvector priority vector.
    """

    _, w = principal_eigenpair(A)
    return w


# =========================================================
# Consistency ratio
# =========================================================

def consistency_ratio(A):
    """
    Compute Saaty's consistency ratio CR.

    CI = (lambda_max - n) / (n - 1)
    CR = CI / RI_n
    """

    n = A.shape[0]

    if n <= 2:
        return 0.0

    lambda_max, _ = principal_eigenpair(A)

    ci = (lambda_max - n) / (n - 1)

    # Avoid tiny negative values due to numerical precision.
    ci = max(0.0, ci)

    ri = RI_VALUES.get(n)

    if ri is None:
        raise ValueError(f"No random index RI available for n={n}")

    if ri == 0:
        return 0.0

    return ci / ri


# =========================================================
# Ranking comparison with explicit tolerance
# =========================================================

def pairwise_order_matrix(weights, tol=RANK_TOL):
    """
    Convert a priority vector into a pairwise order pattern.

    order[i,j] =  1 if i is ranked above j,
    order[i,j] =  0 if i and j are tied within tolerance,
    order[i,j] = -1 if i is ranked below j.

    Two weights are treated as tied whenever

        |w_i - w_j| <= tol * max(1, ||w||_infinity).
    """

    w = np.asarray(weights, dtype=float)
    n = len(w)

    scale = max(1.0, np.max(np.abs(w)))
    eps = tol * scale

    order = np.zeros((n, n), dtype=np.int8)

    for i in range(n):
        for j in range(n):
            diff = w[i] - w[j]

            if diff > eps:
                order[i, j] = 1
            elif diff < -eps:
                order[i, j] = -1
            else:
                order[i, j] = 0

    return order


def rank_reversal_occurred(w_base, w_new, tol=RANK_TOL):
    """
    A rank reversal is recorded if the pairwise order pattern changes.
    """

    base_order = pairwise_order_matrix(w_base, tol=tol)
    new_order = pairwise_order_matrix(w_new, tol=tol)

    return not np.array_equal(base_order, new_order)


def top_set(weights, tol=RANK_TOL):
    """
    Return the set of alternatives tied for first place within tolerance.
    """

    w = np.asarray(weights, dtype=float)

    scale = max(1.0, np.max(np.abs(w)))
    eps = tol * scale

    max_w = np.max(w)

    return set(np.where(max_w - w <= eps)[0])


def top_rank_change_occurred(w_base, w_new, tol=RANK_TOL):
    """
    A change in the top-ranked alternative is recorded if the top set after
    intensification has empty intersection with the baseline top set.
    """

    base_top = top_set(w_base, tol=tol)
    new_top = top_set(w_new, tol=tol)

    return len(base_top.intersection(new_top)) == 0


# =========================================================
# Run one matrix
# =========================================================

def evaluate_one_matrix(O, n, matrix_id):
    """
    Evaluate one ordinal matrix O under the eigenvector method for all alpha.
    """

    rows = []

    A_base = substitute_alpha(O, BASELINE_ALPHA)
    w_base = eigenvector_priority(A_base)
    cr_base = consistency_ratio(A_base)

    cr_class = "CR < 0.10" if cr_base < 0.10 else "CR >= 0.10"

    for alpha in ALPHA_VALUES:

        A_alpha = substitute_alpha(O, alpha)
        w_alpha = eigenvector_priority(A_alpha)

        if alpha == BASELINE_ALPHA:
            rank_reversal = False
            top_change = False
        else:
            rank_reversal = rank_reversal_occurred(w_base, w_alpha)
            top_change = top_rank_change_occurred(w_base, w_alpha)

        rows.append({
            "n": n,
            "matrix_id": matrix_id,
            "method": "EVM",
            "baseline_alpha": BASELINE_ALPHA,
            "alpha": alpha,
            "cr_baseline_alpha_2": cr_base,
            "cr_class": cr_class,
            "rank_reversal_vs_alpha_2": rank_reversal,
            "top_change_vs_alpha_2": top_change,
        })

    return rows


# =========================================================
# Full experiment
# =========================================================

def run_experiment(matrix_bank):
    """
    Run the discrete EVM experiment using the saved random matrices.
    """

    all_rows = []

    print("Running discrete EVM experiment with saved matrices.")
    print(f"Matrix bank: {MATRIX_BANK_PATH}")
    print(f"Ranking tolerance: {RANK_TOL}")
    print(f"Baseline alpha: {BASELINE_ALPHA}")
    print(f"Compared alphas: {ALPHA_VALUES}\n")

    for n in N_VALUES:

        if n not in matrix_bank:
            raise ValueError(f"n={n} not found in the saved matrix bank.")

        matrices_n = matrix_bank[n]

        print(f"n={n}: using {matrices_n.shape[0]} saved matrices")

        for matrix_id, O in enumerate(matrices_n):
            rows = evaluate_one_matrix(O, n, matrix_id)
            all_rows.extend(rows)

    results_df = pd.DataFrame(all_rows)

    return results_df


# =========================================================
# Build reviewer-oriented summaries
# =========================================================

def build_summaries(results_df):
    """
    Build all summaries needed for the revised paper and response letter.
    """

    nonbaseline = results_df[
        results_df["alpha"] != BASELINE_ALPHA
    ].copy()

    # -----------------------------------------------------
    # Matrix-level summary:
    # For each matrix, record whether at least one alpha > 2
    # produced a rank reversal or a top-ranked alternative change.
    # -----------------------------------------------------

    matrix_level = (
        nonbaseline
        .groupby(["n", "matrix_id", "cr_class"])
        .agg(
            any_rank_reversal=("rank_reversal_vs_alpha_2", "any"),
            any_top_change=("top_change_vs_alpha_2", "any"),
            cr_baseline_alpha_2=("cr_baseline_alpha_2", "first"),
        )
        .reset_index()
    )

    # -----------------------------------------------------
    # Summary by n
    # -----------------------------------------------------

    summary_by_n = (
        matrix_level
        .groupby("n")
        .agg(
            matrices=("matrix_id", "count"),
            rank_reversal_count=("any_rank_reversal", "sum"),
            top_change_count=("any_top_change", "sum"),
            mean_cr=("cr_baseline_alpha_2", "mean"),
            median_cr=("cr_baseline_alpha_2", "median"),
        )
        .reset_index()
    )

    summary_by_n["rank_reversal_frequency"] = (
        summary_by_n["rank_reversal_count"] /
        summary_by_n["matrices"]
    )

    summary_by_n["top_change_frequency"] = (
        summary_by_n["top_change_count"] /
        summary_by_n["matrices"]
    )

    # -----------------------------------------------------
    # Summary by n and CR class
    # This directly answers the request:
    # report number of matrices with CR < 0.10 and CR >= 0.10.
    # -----------------------------------------------------

    summary_by_n_cr = (
        matrix_level
        .groupby(["n", "cr_class"])
        .agg(
            matrices=("matrix_id", "count"),
            rank_reversal_count=("any_rank_reversal", "sum"),
            top_change_count=("any_top_change", "sum"),
        )
        .reset_index()
    )

    summary_by_n_cr["rank_reversal_frequency"] = (
        summary_by_n_cr["rank_reversal_count"] /
        summary_by_n_cr["matrices"]
    )

    summary_by_n_cr["top_change_frequency"] = (
        summary_by_n_cr["top_change_count"] /
        summary_by_n_cr["matrices"]
    )

    # -----------------------------------------------------
    # Summary by n and alpha
    # This clarifies that alpha = 3,...,9 are compared with
    # the baseline alpha = 2.
    # -----------------------------------------------------

    summary_by_n_alpha = (
        nonbaseline
        .groupby(["n", "alpha"])
        .agg(
            comparisons=("rank_reversal_vs_alpha_2", "size"),
            rank_reversal_count=("rank_reversal_vs_alpha_2", "sum"),
            top_change_count=("top_change_vs_alpha_2", "sum"),
        )
        .reset_index()
    )

    summary_by_n_alpha["rank_reversal_frequency"] = (
        summary_by_n_alpha["rank_reversal_count"] /
        summary_by_n_alpha["comparisons"]
    )

    summary_by_n_alpha["top_change_frequency"] = (
        summary_by_n_alpha["top_change_count"] /
        summary_by_n_alpha["comparisons"]
    )

    return matrix_level, summary_by_n, summary_by_n_cr, summary_by_n_alpha


# =========================================================
# Save results
# =========================================================

def save_outputs(
    results_df,
    matrix_level,
    summary_by_n,
    summary_by_n_cr,
    summary_by_n_alpha,
):

    excel_path = os.path.join(
        OUTPUT_DIR,
        "discrete_evm_reviewer_ready_results.xlsx"
    )

    with pd.ExcelWriter(excel_path) as writer:

        summary_by_n.to_excel(
            writer,
            sheet_name="summary_by_n",
            index=False
        )

        summary_by_n_cr.to_excel(
            writer,
            sheet_name="summary_by_n_cr",
            index=False
        )

        summary_by_n_alpha.to_excel(
            writer,
            sheet_name="summary_by_n_alpha",
            index=False
        )

        matrix_level.to_excel(
            writer,
            sheet_name="matrix_level",
            index=False
        )

        results_df.to_excel(
            writer,
            sheet_name="alpha_level",
            index=False
        )

    print("\nExcel file saved to:")
    print(excel_path)


# =========================================================
# Plots
# =========================================================

def make_plots(summary_by_n, summary_by_n_alpha):

    # -----------------------------------------------------
    # Figure 1: overall rank reversal frequency by n
    # -----------------------------------------------------

    plt.figure(figsize=(7, 5))

    plt.plot(
        summary_by_n["n"],
        summary_by_n["rank_reversal_frequency"],
        marker="o"
    )

    plt.xlabel("Matrix order n")
    plt.ylabel("IOP rank reversal frequency")
    plt.title("Discrete IOP rank reversal under the eigenvector method")
    plt.xticks(N_VALUES)
    plt.ylim(0, 1)
    plt.tight_layout()

    fig_path = os.path.join(
        OUTPUT_DIR,
        "discrete_evm_rank_reversal_by_n.jpeg"
    )

    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close()

    print("Figure saved to:")
    print(fig_path)

    # -----------------------------------------------------
    # Figure 2: top-ranked alternative change frequency by n
    # -----------------------------------------------------

    plt.figure(figsize=(7, 5))

    plt.plot(
        summary_by_n["n"],
        summary_by_n["top_change_frequency"],
        marker="o"
    )

    plt.xlabel("Matrix order n")
    plt.ylabel("Top-ranked alternative change frequency")
    plt.title("Top-ranked alternative changes under discrete intensification")
    plt.xticks(N_VALUES)
    plt.ylim(0, 1)
    plt.tight_layout()

    fig_path = os.path.join(
        OUTPUT_DIR,
        "discrete_evm_top_change_by_n.jpeg"
    )

    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close()

    print("Figure saved to:")
    print(fig_path)

    # -----------------------------------------------------
    # Figure 3: rank reversal frequency by n and alpha
    # -----------------------------------------------------

    pivot = summary_by_n_alpha.pivot(
        index="n",
        columns="alpha",
        values="rank_reversal_frequency"
    )

    plt.figure(figsize=(8, 5))

    plt.imshow(
        pivot.values,
        aspect="auto",
        origin="lower",
        vmin=0,
        vmax=1
    )

    plt.colorbar(label="Rank reversal frequency")

    plt.xticks(
        ticks=np.arange(len(pivot.columns)),
        labels=pivot.columns
    )

    plt.yticks(
        ticks=np.arange(len(pivot.index)),
        labels=pivot.index
    )

    plt.xlabel("Intensity alpha")
    plt.ylabel("Matrix order n")
    plt.title("Rank reversal frequency relative to alpha = 2")
    plt.tight_layout()

    fig_path = os.path.join(
        OUTPUT_DIR,
        "discrete_evm_rank_reversal_heatmap_alpha.jpeg"
    )

    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close()

    print("Figure saved to:")
    print(fig_path)


# =========================================================
# Main
# =========================================================

if __name__ == "__main__":

    matrix_bank = load_matrix_bank()

    results_df = run_experiment(matrix_bank)

    (
        matrix_level,
        summary_by_n,
        summary_by_n_cr,
        summary_by_n_alpha,
    ) = build_summaries(results_df)

    print("\nSummary by n:")
    print(summary_by_n)

    print("\nSummary by n and CR class:")
    print(summary_by_n_cr)

    print("\nSummary by n and alpha:")
    print(summary_by_n_alpha)

    save_outputs(
        results_df,
        matrix_level,
        summary_by_n,
        summary_by_n_cr,
        summary_by_n_alpha,
    )

    make_plots(summary_by_n, summary_by_n_alpha)

In [ ]:
#Experimento 2
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.linalg import eig

# =========================================================
# Configuration
# =========================================================

SEED = 42
TRIALS_PER_N = 5000

N_VALUES = list(range(3, 10))          # n = 3,...,9
ALPHA_VALUES = list(range(2, 10))      # alpha = 2,...,9
BASELINE_ALPHA = 2
RANK_TOL = 1e-10

# Folder where the saved random matrices are stored
INPUT_DIR = "discrete_shared_matrices"

# New output folder
OUTPUT_DIR = "discrete 2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MATRIX_BANK_PATH = os.path.join(
    INPUT_DIR,
    f"ordinal_matrix_bank_seed_{SEED}_trials_{TRIALS_PER_N}.npz"
)

RI_VALUES = {
    1: 0.00,
    2: 0.00,
    3: 0.58,
    4: 0.90,
    5: 1.12,
    6: 1.24,
    7: 1.32,
    8: 1.41,
    9: 1.45,
    10: 1.49,
}


# =========================================================
# Load saved ordinal matrices
# =========================================================

def load_matrix_bank():
    """
    Load the ordinal matrices generated once and reused across experiments.

    The stored ordinal matrix O is encoded as:
        O[i,j] =  1  -> strict preference P
        O[i,j] =  0  -> tie
        O[i,j] = -1  -> reciprocal strict preference 1/P

    For a given intensity alpha:
        A(alpha) = alpha ** O
    """

    if not os.path.exists(MATRIX_BANK_PATH):
        raise FileNotFoundError(
            f"Matrix bank not found:\n{MATRIX_BANK_PATH}\n\n"
            "Run the matrix-generation script first."
        )

    data = np.load(MATRIX_BANK_PATH)

    matrix_bank = {}

    for key in data.files:
        n = int(key.split("_")[1])
        matrix_bank[n] = data[key]

    return matrix_bank


def substitute_alpha(O, alpha):
    """
    Convert ordinal exponent matrix O into a cardinal reciprocal PCM.
    """
    return alpha ** O.astype(float)


# =========================================================
# Eigenvector method
# =========================================================

def principal_eigenpair(A):
    """
    Principal eigenvalue and Perron priority vector.
    """

    eigenvalues, eigenvectors = eig(A)
    idx = np.argmax(eigenvalues.real)

    lambda_max = eigenvalues[idx].real
    w = eigenvectors[:, idx].real

    # Eigenvectors have arbitrary sign
    if np.sum(w) < 0:
        w = -w

    # Numerical safeguard
    if np.any(w <= 0):
        w = np.abs(w)

    w = w / np.sum(w)

    return lambda_max, w


def eigenvector_priority(A):
    """
    Eigenvector priority vector.
    """
    _, w = principal_eigenpair(A)
    return w


# =========================================================
# Saaty's consistency ratio
# =========================================================

def consistency_ratio(A):
    """
    Compute Saaty's consistency ratio CR at the given cardinal PCM.

    CI = (lambda_max - n) / (n - 1)
    CR = CI / RI_n
    """

    n = A.shape[0]

    if n <= 2:
        return 0.0

    lambda_max, _ = principal_eigenpair(A)

    ci = (lambda_max - n) / (n - 1)

    # Avoid tiny negative numerical artefacts
    ci = max(0.0, ci)

    ri = RI_VALUES.get(n)

    if ri is None:
        raise ValueError(f"No RI value available for n={n}")

    if ri == 0:
        return 0.0

    return ci / ri


# =========================================================
# Ranking comparison with explicit tolerance
# =========================================================

def pairwise_order_matrix(weights, tol=RANK_TOL):
    """
    Pairwise order pattern induced by a priority vector.

    order[i,j] =  1 if i is ranked above j
    order[i,j] =  0 if i and j are tied within tolerance
    order[i,j] = -1 if i is ranked below j

    Tie rule:
        |w_i - w_j| <= tol * max(1, ||w||_infinity)
    """

    w = np.asarray(weights, dtype=float)
    n = len(w)

    scale = max(1.0, np.max(np.abs(w)))
    eps = tol * scale

    order = np.zeros((n, n), dtype=np.int8)

    for i in range(n):
        for j in range(n):

            diff = w[i] - w[j]

            if diff > eps:
                order[i, j] = 1
            elif diff < -eps:
                order[i, j] = -1
            else:
                order[i, j] = 0

    return order


def rank_reversal_occurred(w_base, w_new, tol=RANK_TOL):
    """
    Rank reversal occurs if the pairwise order pattern changes.
    """

    return not np.array_equal(
        pairwise_order_matrix(w_base, tol),
        pairwise_order_matrix(w_new, tol)
    )


def top_set(weights, tol=RANK_TOL):
    """
    Set of top-ranked alternatives, allowing ties within tolerance.
    """

    w = np.asarray(weights, dtype=float)

    scale = max(1.0, np.max(np.abs(w)))
    eps = tol * scale

    max_w = np.max(w)

    return set(np.where(max_w - w <= eps)[0])


def top_rank_change_occurred(w_base, w_new, tol=RANK_TOL):
    """
    Top-ranked alternative changes if the top set after intensification
    has empty intersection with the baseline top set.
    """

    base_top = top_set(w_base, tol)
    new_top = top_set(w_new, tol)

    return len(base_top.intersection(new_top)) == 0


# =========================================================
# Evaluate one matrix
# =========================================================

def evaluate_one_matrix(O, n, matrix_id):
    """
    For one ordinal matrix O:
    - compute CR at alpha = 2;
    - compare the baseline ranking at alpha = 2 with alpha = 3,...,9;
    - record whether any rank reversal occurs;
    - record whether the top-ranked alternative changes.
    """

    A_base = substitute_alpha(O, BASELINE_ALPHA)

    cr = consistency_ratio(A_base)
    cr_group = "CR < 0.10" if cr < 0.10 else "CR >= 0.10"

    w_base = eigenvector_priority(A_base)

    any_rank_reversal = False
    any_top_change = False

    alpha_rows = []

    for alpha in ALPHA_VALUES:

        A_alpha = substitute_alpha(O, alpha)
        w_alpha = eigenvector_priority(A_alpha)

        if alpha == BASELINE_ALPHA:
            rank_reversal = False
            top_change = False
        else:
            rank_reversal = rank_reversal_occurred(w_base, w_alpha)
            top_change = top_rank_change_occurred(w_base, w_alpha)

        any_rank_reversal = any_rank_reversal or rank_reversal
        any_top_change = any_top_change or top_change

        alpha_rows.append({
            "n": n,
            "matrix_id": matrix_id,
            "alpha": alpha,
            "cr_baseline_alpha_2": cr,
            "cr_group": cr_group,
            "rank_reversal_vs_alpha_2": rank_reversal,
            "top_change_vs_alpha_2": top_change,
        })

    matrix_row = {
        "n": n,
        "matrix_id": matrix_id,
        "cr_baseline_alpha_2": cr,
        "cr_group": cr_group,
        "any_rank_reversal": any_rank_reversal,
        "any_top_change": any_top_change,
    }

    return matrix_row, alpha_rows


# =========================================================
# Run CR analysis using saved matrices
# =========================================================

def run_cr_analysis(matrix_bank):

    matrix_rows = []
    alpha_rows = []

    print("Running discrete CR analysis using saved matrices.")
    print(f"Matrix bank: {MATRIX_BANK_PATH}")
    print(f"Output folder: {OUTPUT_DIR}")
    print(f"Baseline alpha: {BASELINE_ALPHA}")
    print(f"Compared alphas: {ALPHA_VALUES}")
    print(f"Ranking tolerance: {RANK_TOL}\n")

    for n in N_VALUES:

        if n not in matrix_bank:
            raise ValueError(f"n={n} not found in matrix bank.")

        matrices_n = matrix_bank[n]

        print(f"n={n}: using {matrices_n.shape[0]} saved matrices")

        for matrix_id, O in enumerate(matrices_n):

            matrix_row, rows_alpha = evaluate_one_matrix(O, n, matrix_id)

            matrix_rows.append(matrix_row)
            alpha_rows.extend(rows_alpha)

    matrix_df = pd.DataFrame(matrix_rows)
    alpha_df = pd.DataFrame(alpha_rows)

    return matrix_df, alpha_df


# =========================================================
# Build summaries
# =========================================================

def build_summaries(matrix_df, alpha_df):

    # -----------------------------------------------------
    # Summary by n and CR group
    # -----------------------------------------------------

    summary_by_n_cr = (
        matrix_df
        .groupby(["n", "cr_group"])
        .agg(
            matrices=("matrix_id", "count"),
            rank_reversal_count=("any_rank_reversal", "sum"),
            top_change_count=("any_top_change", "sum"),
            mean_cr=("cr_baseline_alpha_2", "mean"),
            median_cr=("cr_baseline_alpha_2", "median"),
        )
        .reset_index()
    )

    summary_by_n_cr["rank_reversal_frequency"] = (
        summary_by_n_cr["rank_reversal_count"] /
        summary_by_n_cr["matrices"]
    )

    summary_by_n_cr["top_change_frequency"] = (
        summary_by_n_cr["top_change_count"] /
        summary_by_n_cr["matrices"]
    )

    # -----------------------------------------------------
    # Overall summary by n
    # -----------------------------------------------------

    summary_by_n = (
        matrix_df
        .groupby("n")
        .agg(
            matrices=("matrix_id", "count"),
            rank_reversal_count=("any_rank_reversal", "sum"),
            top_change_count=("any_top_change", "sum"),
            mean_cr=("cr_baseline_alpha_2", "mean"),
            median_cr=("cr_baseline_alpha_2", "median"),
        )
        .reset_index()
    )

    summary_by_n["rank_reversal_frequency"] = (
        summary_by_n["rank_reversal_count"] /
        summary_by_n["matrices"]
    )

    summary_by_n["top_change_frequency"] = (
        summary_by_n["top_change_count"] /
        summary_by_n["matrices"]
    )

    # -----------------------------------------------------
    # Summary by n, CR group and alpha
    # -----------------------------------------------------

    alpha_nonbaseline = alpha_df[alpha_df["alpha"] != BASELINE_ALPHA].copy()

    summary_by_n_cr_alpha = (
        alpha_nonbaseline
        .groupby(["n", "cr_group", "alpha"])
        .agg(
            comparisons=("rank_reversal_vs_alpha_2", "size"),
            rank_reversal_count=("rank_reversal_vs_alpha_2", "sum"),
            top_change_count=("top_change_vs_alpha_2", "sum"),
        )
        .reset_index()
    )

    summary_by_n_cr_alpha["rank_reversal_frequency"] = (
        summary_by_n_cr_alpha["rank_reversal_count"] /
        summary_by_n_cr_alpha["comparisons"]
    )

    summary_by_n_cr_alpha["top_change_frequency"] = (
        summary_by_n_cr_alpha["top_change_count"] /
        summary_by_n_cr_alpha["comparisons"]
    )

    return summary_by_n, summary_by_n_cr, summary_by_n_cr_alpha


# =========================================================
# Save outputs
# =========================================================

def save_outputs(
    matrix_df,
    alpha_df,
    summary_by_n,
    summary_by_n_cr,
    summary_by_n_cr_alpha,
):

    excel_path = os.path.join(
        OUTPUT_DIR,
        "discrete_cr_analysis_same_matrices.xlsx"
    )

    with pd.ExcelWriter(excel_path) as writer:

        summary_by_n.to_excel(
            writer,
            sheet_name="summary_by_n",
            index=False
        )

        summary_by_n_cr.to_excel(
            writer,
            sheet_name="summary_by_n_cr",
            index=False
        )

        summary_by_n_cr_alpha.to_excel(
            writer,
            sheet_name="summary_n_cr_alpha",
            index=False
        )

        matrix_df.to_excel(
            writer,
            sheet_name="matrix_level",
            index=False
        )

        alpha_df.to_excel(
            writer,
            sheet_name="alpha_level",
            index=False
        )

    print("\nExcel file saved to:")
    print(excel_path)


# =========================================================
# Plots
# =========================================================

def plot_cr_reversal_all_n(summary_by_n_cr):

    pivot = summary_by_n_cr.pivot(
        index="n",
        columns="cr_group",
        values="rank_reversal_frequency"
    )

    pivot = pivot.reindex(columns=["CR < 0.10", "CR >= 0.10"])

    ax = pivot.plot(kind="bar", figsize=(8, 5))

    ax.set_xlabel("Matrix order n")
    ax.set_ylabel("IOP rank reversal frequency")
    ax.set_title("Discrete IOP rank reversal by consistency-ratio group")
    ax.set_ylim(0, 1)

    plt.xticks(rotation=0)
    plt.tight_layout()

    fig_path = os.path.join(
        OUTPUT_DIR,
        "discrete_iop_rank_reversal_vs_cr_all_n.jpeg"
    )

    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close()

    print("Figure saved to:")
    print(fig_path)


def plot_top_change_all_n(summary_by_n_cr):

    pivot = summary_by_n_cr.pivot(
        index="n",
        columns="cr_group",
        values="top_change_frequency"
    )

    pivot = pivot.reindex(columns=["CR < 0.10", "CR >= 0.10"])

    ax = pivot.plot(kind="bar", figsize=(8, 5))

    ax.set_xlabel("Matrix order n")
    ax.set_ylabel("Top-ranked alternative change frequency")
    ax.set_title("Top-ranked alternative changes by consistency-ratio group")
    ax.set_ylim(0, 1)

    plt.xticks(rotation=0)
    plt.tight_layout()

    fig_path = os.path.join(
        OUTPUT_DIR,
        "discrete_top_change_vs_cr_all_n.jpeg"
    )

    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close()

    print("Figure saved to:")
    print(fig_path)


def plot_original_n7_style(summary_by_n_cr, n_selected=7):

    df_n = summary_by_n_cr[summary_by_n_cr["n"] == n_selected].copy()

    if df_n.empty:
        print(f"No data available for n={n_selected}")
        return

    df_n = df_n.set_index("cr_group")
    df_n = df_n.reindex(["CR < 0.10", "CR >= 0.10"])

    ax = df_n["rank_reversal_frequency"].plot(
        kind="bar",
        figsize=(6, 4)
    )

    ax.set_xlabel("Consistency Ratio (CR)")
    ax.set_ylabel("IOP rank reversal frequency")
    ax.set_title(f"Discrete IOP rank reversal vs inconsistency, n={n_selected}")
    ax.set_ylim(0, 1)

    plt.xticks(rotation=0)
    plt.tight_layout()

    fig_path = os.path.join(
        OUTPUT_DIR,
        f"discrete_iop_rank_reversal_vs_cr_n{n_selected}.jpeg"
    )

    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close()

    print("Figure saved to:")
    print(fig_path)


# =========================================================
# Main
# =========================================================

if __name__ == "__main__":

    matrix_bank = load_matrix_bank()

    matrix_df, alpha_df = run_cr_analysis(matrix_bank)

    (
        summary_by_n,
        summary_by_n_cr,
        summary_by_n_cr_alpha,
    ) = build_summaries(matrix_df, alpha_df)

    print("\nOverall summary by n:")
    print(summary_by_n)

    print("\nSummary by n and CR group:")
    print(summary_by_n_cr)

    print("\nSummary by n, CR group and alpha:")
    print(summary_by_n_cr_alpha)

    save_outputs(
        matrix_df,
        alpha_df,
        summary_by_n,
        summary_by_n_cr,
        summary_by_n_cr_alpha,
    )

    plot_cr_reversal_all_n(summary_by_n_cr)
    plot_top_change_all_n(summary_by_n_cr)
    plot_original_n7_style(summary_by_n_cr, n_selected=7)

In [2]:
# Experimento 3: discrete comparison across retained methods
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from numpy.linalg import eig
from scipy.optimize import minimize
from matplotlib.ticker import PercentFormatter


# =========================================================
# Configuration
# =========================================================

SEED = 42
TRIALS_PER_N = 5000

N_VALUES = list(range(3, 10))          # n = 3,...,9
ALPHA_VALUES = list(range(2, 10))      # alpha = 2,...,9
BASELINE_ALPHA = 2
RANK_TOL = 1e-10

# Bound for the optimisation-based CMM.
# It prevents numerical overflow in exp(x_i - x_j).
LOG_WEIGHT_BOUND = 8.0
OPT_MAXITER = 300

INPUT_DIR = "discrete_shared_matrices"
OUTPUT_DIR = "results_3_retained_methods"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MATRIX_BANK_PATH = os.path.join(
    INPUT_DIR,
    f"ordinal_matrix_bank_seed_{SEED}_trials_{TRIALS_PER_N}.npz"
)

RI_VALUES = {
    1: 0.00,
    2: 0.00,
    3: 0.58,
    4: 0.90,
    5: 1.12,
    6: 1.24,
    7: 1.32,
    8: 1.41,
    9: 1.45,
    10: 1.49,
}

METHOD_ORDER = ["EVM", "RSM", "CSM", "HMM", "CMM"]

MARKERS = {
    "EVM": "o",
    "RSM": "s",
    "CSM": "^",
    "HMM": "D",
    "CMM": "x",
}

LINESTYLES = {
    "EVM": "-",
    "RSM": "--",
    "CSM": "-.",
    "HMM": ":",
    "CMM": "-",
}


# =========================================================
# Load saved ordinal matrices
# =========================================================

def load_matrix_bank():
    if not os.path.exists(MATRIX_BANK_PATH):
        raise FileNotFoundError(
            f"Matrix bank not found:\n{MATRIX_BANK_PATH}\n\n"
            "Run the matrix-generation script first."
        )

    data = np.load(MATRIX_BANK_PATH)
    matrix_bank = {}

    for key in data.files:
        n = int(key.split("_")[1])
        matrix_bank[n] = data[key]

    return matrix_bank


def substitute_alpha(O, alpha):
    return alpha ** O.astype(float)


# =========================================================
# Utility functions
# =========================================================

def normalize_positive(w):
    w = np.asarray(w, dtype=float)

    if np.any(~np.isfinite(w)):
        raise ValueError("Priority vector contains non-finite values.")

    if np.any(w < 0):
        raise ValueError("Priority vector contains negative values.")

    total = np.sum(w)

    if total <= 0:
        raise ValueError("Priority vector has non-positive sum.")

    return w / total


def geometric_mean_start(A):
    gm = np.prod(A, axis=1) ** (1.0 / A.shape[0])
    gm = normalize_positive(gm)
    return np.log(gm)


def fixed_scale_log_vector(z):
    """
    For ratio-based optimisation, fix the last log-weight to zero.
    """
    z = np.asarray(z, dtype=float)
    z = np.clip(z, -LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)
    return np.concatenate([z, np.array([0.0])])


def ratio_matrix_from_log_vector(x):
    """
    Safely compute R_ij = exp(x_i - x_j), avoiding overflow.
    """
    x = np.asarray(x, dtype=float)
    d = x[:, None] - x[None, :]
    d = np.clip(d, -2 * LOG_WEIGHT_BOUND, 2 * LOG_WEIGHT_BOUND)
    return np.exp(d)


# =========================================================
# Priority derivation methods
# =========================================================

def principal_eigenpair(A):
    eigenvalues, eigenvectors = eig(A)
    idx = np.argmax(eigenvalues.real)

    lambda_max = eigenvalues[idx].real
    w = eigenvectors[:, idx].real

    if np.sum(w) < 0:
        w = -w

    if np.any(w <= 0):
        w = np.abs(w)

    w = normalize_positive(w)

    return lambda_max, w


def eigenvector_priority(A):
    _, w = principal_eigenpair(A)
    return w


def row_sum_priority(A):
    rs = np.sum(A, axis=1)
    return normalize_positive(rs)


def column_sum_priority(A):
    col_sums = np.sum(A, axis=0)
    norm_matrix = A / col_sums
    w = np.sum(norm_matrix, axis=1)
    return normalize_positive(w)


def harmonic_mean_priority(A):
    n = A.shape[0]
    hm = n / np.sum(1.0 / A, axis=1)
    return normalize_positive(hm)


def cosine_maximization_priority(A):
    """
    Cosine maximisation method.

    The method solves:
        max cosine(A, [w_i/w_j])
    in bounded log-weight variables with the last log-weight fixed to zero.
    """

    n = A.shape[0]
    norm_A = np.linalg.norm(A)

    if norm_A <= 0:
        raise ValueError("Invalid matrix norm.")

    log_start = geometric_mean_start(A)
    z0 = log_start[:-1] - log_start[-1]
    z0 = np.clip(z0, -LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)

    bounds = [(-LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)] * (n - 1)

    def objective(z):
        x = fixed_scale_log_vector(z)
        R = ratio_matrix_from_log_vector(x)

        numerator = np.sum(A * R)
        norm_R = np.linalg.norm(R)

        if norm_R <= 0 or not np.isfinite(norm_R):
            return 1e100

        cosine = numerator / (norm_A * norm_R)

        if not np.isfinite(cosine):
            return 1e100

        return -cosine

    res = minimize(
        objective,
        z0,
        method="L-BFGS-B",
        bounds=bounds,
        options={
            "maxiter": OPT_MAXITER,
            "ftol": 1e-10,
        }
    )

    if not res.success:
        raise RuntimeError(f"CMM optimisation did not converge: {res.message}")

    x = fixed_scale_log_vector(res.x)
    w = np.exp(x - np.max(x))

    return normalize_positive(w)


# =========================================================
# Consistency ratio
# =========================================================

def consistency_ratio(A):
    n = A.shape[0]

    if n <= 2:
        return 0.0

    lambda_max, _ = principal_eigenpair(A)

    ci = (lambda_max - n) / (n - 1)
    ci = max(0.0, ci)

    ri = RI_VALUES.get(n)

    if ri is None:
        raise ValueError(f"No RI value available for n={n}")

    if ri == 0:
        return 0.0

    return ci / ri


# =========================================================
# Ranking comparison with explicit tolerance
# =========================================================

def pairwise_order_matrix(weights, tol=RANK_TOL):
    w = np.asarray(weights, dtype=float)
    n = len(w)

    scale = max(1.0, np.max(np.abs(w)))
    eps = tol * scale

    order = np.zeros((n, n), dtype=np.int8)

    for i in range(n):
        for j in range(n):
            diff = w[i] - w[j]

            if diff > eps:
                order[i, j] = 1
            elif diff < -eps:
                order[i, j] = -1
            else:
                order[i, j] = 0

    return order


def rank_reversal_occurred(w_base, w_new, tol=RANK_TOL):
    return not np.array_equal(
        pairwise_order_matrix(w_base, tol),
        pairwise_order_matrix(w_new, tol)
    )


def top_set(weights, tol=RANK_TOL):
    w = np.asarray(weights, dtype=float)

    scale = max(1.0, np.max(np.abs(w)))
    eps = tol * scale

    max_w = np.max(w)

    return set(np.where(max_w - w <= eps)[0])


def top_rank_change_occurred(w_base, w_new, tol=RANK_TOL):
    base_top = top_set(w_base, tol)
    new_top = top_set(w_new, tol)

    return len(base_top.intersection(new_top)) == 0


# =========================================================
# Methods included
# =========================================================

PRIORITY_METHODS = {
    "EVM": eigenvector_priority,
    "RSM": row_sum_priority,
    "CSM": column_sum_priority,
    "HMM": harmonic_mean_priority,
    "CMM": cosine_maximization_priority,
}

PLOT_METHODS = METHOD_ORDER


# =========================================================
# Evaluate one method on one matrix
# =========================================================

def evaluate_method_on_matrix(method_name, method_func, O, n, matrix_id):
    rows = []

    A_base = substitute_alpha(O, BASELINE_ALPHA)

    cr = consistency_ratio(A_base)
    cr_group = "CR < 0.10" if cr < 0.10 else "CR >= 0.10"

    try:
        w_base = method_func(A_base)
        baseline_success = True
        baseline_error = ""
    except Exception as exc:
        w_base = None
        baseline_success = False
        baseline_error = str(exc)

    for alpha in ALPHA_VALUES:

        if not baseline_success:
            rows.append({
                "method": method_name,
                "n": n,
                "matrix_id": matrix_id,
                "baseline_alpha": BASELINE_ALPHA,
                "alpha": alpha,
                "cr_baseline_alpha_2": cr,
                "cr_group": cr_group,
                "success": False,
                "error": baseline_error,
                "rank_reversal_vs_alpha_2": np.nan,
                "top_change_vs_alpha_2": np.nan,
            })
            continue

        A_alpha = substitute_alpha(O, alpha)

        try:
            w_alpha = method_func(A_alpha)

            if alpha == BASELINE_ALPHA:
                rank_reversal = False
                top_change = False
            else:
                rank_reversal = rank_reversal_occurred(w_base, w_alpha)
                top_change = top_rank_change_occurred(w_base, w_alpha)

            rows.append({
                "method": method_name,
                "n": n,
                "matrix_id": matrix_id,
                "baseline_alpha": BASELINE_ALPHA,
                "alpha": alpha,
                "cr_baseline_alpha_2": cr,
                "cr_group": cr_group,
                "success": True,
                "error": "",
                "rank_reversal_vs_alpha_2": rank_reversal,
                "top_change_vs_alpha_2": top_change,
            })

        except Exception as exc:
            rows.append({
                "method": method_name,
                "n": n,
                "matrix_id": matrix_id,
                "baseline_alpha": BASELINE_ALPHA,
                "alpha": alpha,
                "cr_baseline_alpha_2": cr,
                "cr_group": cr_group,
                "success": False,
                "error": str(exc),
                "rank_reversal_vs_alpha_2": np.nan,
                "top_change_vs_alpha_2": np.nan,
            })

    return rows


# =========================================================
# Run full experiment
# =========================================================

def run_experiment(matrix_bank):
    all_rows = []

    print("Running discrete comparison across retained methods.")
    print(f"Matrix bank: {MATRIX_BANK_PATH}")
    print(f"Output folder: {OUTPUT_DIR}")
    print(f"n values: {N_VALUES}")
    print(f"alpha values: {ALPHA_VALUES}")
    print(f"baseline alpha: {BASELINE_ALPHA}")
    print(f"ranking tolerance: {RANK_TOL}")
    print(f"log-weight bound: {LOG_WEIGHT_BOUND}")

    print("\nMethods included:")
    for method_name in PRIORITY_METHODS:
        print(f"- {method_name}")

    for n in N_VALUES:

        if n not in matrix_bank:
            raise ValueError(f"n={n} not found in matrix bank.")

        matrices_n = matrix_bank[n]

        print(f"\nRunning n={n}; matrices={matrices_n.shape[0]}")

        for matrix_id, O in enumerate(matrices_n):

            for method_name, method_func in PRIORITY_METHODS.items():

                rows = evaluate_method_on_matrix(
                    method_name=method_name,
                    method_func=method_func,
                    O=O,
                    n=n,
                    matrix_id=matrix_id,
                )

                all_rows.extend(rows)

    results_df = pd.DataFrame(all_rows)

    return results_df


# =========================================================
# Build summaries
# =========================================================

def build_summaries(results_df):

    successful = results_df[results_df["success"]].copy()

    nonbaseline = successful[successful["alpha"] != BASELINE_ALPHA].copy()

    matrix_level = (
        nonbaseline
        .groupby(["method", "n", "matrix_id", "cr_group"])
        .agg(
            any_rank_reversal=("rank_reversal_vs_alpha_2", "any"),
            any_top_change=("top_change_vs_alpha_2", "any"),
            cr_baseline_alpha_2=("cr_baseline_alpha_2", "first"),
        )
        .reset_index()
    )

    summary_by_method_n = (
        matrix_level
        .groupby(["method", "n"])
        .agg(
            matrices=("matrix_id", "count"),
            rank_reversal_count=("any_rank_reversal", "sum"),
            top_change_count=("any_top_change", "sum"),
            mean_cr=("cr_baseline_alpha_2", "mean"),
            median_cr=("cr_baseline_alpha_2", "median"),
        )
        .reset_index()
    )

    summary_by_method_n["rank_reversal_frequency"] = (
        summary_by_method_n["rank_reversal_count"] /
        summary_by_method_n["matrices"]
    )

    summary_by_method_n["top_change_frequency"] = (
        summary_by_method_n["top_change_count"] /
        summary_by_method_n["matrices"]
    )

    average_by_method = (
        summary_by_method_n
        .groupby("method")
        .agg(
            average_rank_reversal_frequency=("rank_reversal_frequency", "mean"),
            average_top_change_frequency=("top_change_frequency", "mean"),
            total_matrices=("matrices", "sum"),
        )
        .reset_index()
    )

    summary_by_method_n_alpha = (
        nonbaseline
        .groupby(["method", "n", "alpha"])
        .agg(
            comparisons=("rank_reversal_vs_alpha_2", "size"),
            rank_reversal_count=("rank_reversal_vs_alpha_2", "sum"),
            top_change_count=("top_change_vs_alpha_2", "sum"),
        )
        .reset_index()
    )

    summary_by_method_n_alpha["rank_reversal_frequency"] = (
        summary_by_method_n_alpha["rank_reversal_count"] /
        summary_by_method_n_alpha["comparisons"]
    )

    summary_by_method_n_alpha["top_change_frequency"] = (
        summary_by_method_n_alpha["top_change_count"] /
        summary_by_method_n_alpha["comparisons"]
    )

    summary_by_method_n_cr = (
        matrix_level
        .groupby(["method", "n", "cr_group"])
        .agg(
            matrices=("matrix_id", "count"),
            rank_reversal_count=("any_rank_reversal", "sum"),
            top_change_count=("any_top_change", "sum"),
        )
        .reset_index()
    )

    summary_by_method_n_cr["rank_reversal_frequency"] = (
        summary_by_method_n_cr["rank_reversal_count"] /
        summary_by_method_n_cr["matrices"]
    )

    summary_by_method_n_cr["top_change_frequency"] = (
        summary_by_method_n_cr["top_change_count"] /
        summary_by_method_n_cr["matrices"]
    )

    failure_summary = (
        results_df[~results_df["success"]]
        .groupby(["method", "n", "alpha", "error"])
        .size()
        .reset_index(name="failures")
    )

    return (
        matrix_level,
        summary_by_method_n,
        average_by_method,
        summary_by_method_n_alpha,
        summary_by_method_n_cr,
        failure_summary,
    )


# =========================================================
# Save outputs
# =========================================================

def save_outputs(
    results_df,
    matrix_level,
    summary_by_method_n,
    average_by_method,
    summary_by_method_n_alpha,
    summary_by_method_n_cr,
    failure_summary,
):

    excel_path = os.path.join(
        OUTPUT_DIR,
        "discrete_comparison_retained_methods_reviewer_ready.xlsx"
    )

    with pd.ExcelWriter(excel_path) as writer:

        average_by_method.to_excel(
            writer,
            sheet_name="average_by_method",
            index=False
        )

        summary_by_method_n.to_excel(
            writer,
            sheet_name="summary_method_n",
            index=False
        )

        summary_by_method_n_alpha.to_excel(
            writer,
            sheet_name="summary_method_n_alpha",
            index=False
        )

        summary_by_method_n_cr.to_excel(
            writer,
            sheet_name="summary_method_n_cr",
            index=False
        )

        matrix_level.to_excel(
            writer,
            sheet_name="matrix_level",
            index=False
        )

        failure_summary.to_excel(
            writer,
            sheet_name="failures",
            index=False
        )

        results_df.to_excel(
            writer,
            sheet_name="alpha_level",
            index=False
        )

    print("\nExcel file saved to:")
    print(excel_path)

    latex_path = os.path.join(
        OUTPUT_DIR,
        "table_average_discrete_comparison_retained_methods.tex"
    )

    latex_df = average_by_method.copy()
    latex_df = latex_df[latex_df["method"].isin(PLOT_METHODS)]
    latex_df = latex_df.set_index("method").loc[METHOD_ORDER].reset_index()

    latex_df["average_rank_reversal_frequency"] *= 100
    latex_df["average_top_change_frequency"] *= 100

    latex_df = latex_df.rename(columns={
        "method": "Method",
        "average_rank_reversal_frequency": "Average rank-reversal frequency (\\%)",
        "average_top_change_frequency": "Average top-change frequency (\\%)",
        "total_matrices": "Total matrices",
    })

    latex_table = latex_df.to_latex(
        index=False,
        float_format="%.1f",
        caption=(
            "Average discrete IOP rank-reversal frequency and top-ranked "
            "alternative change frequency across the retained priority "
            "derivation methods."
        ),
        label="tab:discrete_comparison_methods_revised",
        escape=False,
    )

    with open(latex_path, "w", encoding="utf-8") as f:
        f.write(latex_table)

    print("LaTeX table saved to:")
    print(latex_path)


# =========================================================
# Plots
# =========================================================

def save_figure(fig, filename_base):
    jpeg_path = os.path.join(OUTPUT_DIR, f"{filename_base}.jpeg")
    pdf_path = os.path.join(OUTPUT_DIR, f"{filename_base}.pdf")

    fig.savefig(jpeg_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")

    print("Figure saved to:")
    print(jpeg_path)
    print(pdf_path)


def plot_average_rank_and_top_grouped(average_by_method):
    plot_df = average_by_method[
        average_by_method["method"].isin(PLOT_METHODS)
    ].copy()

    plot_df = plot_df.set_index("method").loc[METHOD_ORDER].reset_index()

    methods = plot_df["method"].to_numpy()
    x = np.arange(len(methods))
    width = 0.36

    rr = plot_df["average_rank_reversal_frequency"].to_numpy()
    top = plot_df["average_top_change_frequency"].to_numpy()

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.bar(
        x - width / 2,
        rr,
        width,
        label="Rank reversal",
        hatch="//",
        edgecolor="black",
    )

    ax.bar(
        x + width / 2,
        top,
        width,
        label="Top-ranked alternative change",
        hatch="\\\\",
        edgecolor="black",
    )

    ax.set_xlabel("Priority derivation method")
    ax.set_ylabel("Average frequency")
    ax.set_title("Discrete IOP rank reversal and top-ranked alternative changes")
    ax.set_xticks(x)
    ax.set_xticklabels(methods)
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    ax.legend(frameon=False)

    plt.tight_layout()

    save_figure(
        fig,
        "discrete_bar_rank_reversal_top_change_retained_methods"
    )

    plt.close(fig)


def plot_rank_reversal_by_n(summary_by_method_n):
    plot_df = summary_by_method_n[
        summary_by_method_n["method"].isin(PLOT_METHODS)
    ].copy()

    fig, ax = plt.subplots(figsize=(9, 5))

    for method in METHOD_ORDER:

        df_m = plot_df[plot_df["method"] == method]

        ax.plot(
            df_m["n"],
            df_m["rank_reversal_frequency"],
            marker=MARKERS.get(method, "o"),
            linestyle=LINESTYLES.get(method, "-"),
            label=method
        )

    ax.set_xlabel(r"Matrix order $n$")
    ax.set_ylabel("IOP rank-reversal frequency")
    ax.set_title("Discrete IOP rank reversal by matrix order")
    ax.set_xticks(N_VALUES)
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    ax.legend(frameon=False)

    plt.tight_layout()

    save_figure(
        fig,
        "discrete_rank_reversal_by_n_retained_methods"
    )

    plt.close(fig)


def plot_top_change_by_n(summary_by_method_n):
    plot_df = summary_by_method_n[
        summary_by_method_n["method"].isin(PLOT_METHODS)
    ].copy()

    fig, ax = plt.subplots(figsize=(9, 5))

    for method in METHOD_ORDER:

        df_m = plot_df[plot_df["method"] == method]

        ax.plot(
            df_m["n"],
            df_m["top_change_frequency"],
            marker=MARKERS.get(method, "o"),
            linestyle=LINESTYLES.get(method, "-"),
            label=method
        )

    ax.set_xlabel(r"Matrix order $n$")
    ax.set_ylabel("Top-ranked alternative change frequency")
    ax.set_title("Top-ranked alternative changes by matrix order")
    ax.set_xticks(N_VALUES)
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    ax.legend(frameon=False)

    plt.tight_layout()

    save_figure(
        fig,
        "discrete_top_change_by_n_retained_methods"
    )

    plt.close(fig)


# =========================================================
# Main
# =========================================================

if __name__ == "__main__":

    matrix_bank = load_matrix_bank()

    results_df = run_experiment(matrix_bank)

    (
        matrix_level,
        summary_by_method_n,
        average_by_method,
        summary_by_method_n_alpha,
        summary_by_method_n_cr,
        failure_summary,
    ) = build_summaries(results_df)

    print("\nAverage by method:")
    print(average_by_method)

    print("\nSummary by method and n:")
    print(summary_by_method_n)

    print("\nSummary by method, n and CR group:")
    print(summary_by_method_n_cr)

    if not failure_summary.empty:
        print("\nFailures:")
        print(failure_summary)
    else:
        print("\nNo method failures recorded.")

    save_outputs(
        results_df,
        matrix_level,
        summary_by_method_n,
        average_by_method,
        summary_by_method_n_alpha,
        summary_by_method_n_cr,
        failure_summary,
    )

    plot_average_rank_and_top_grouped(average_by_method)
    plot_rank_reversal_by_n(summary_by_method_n)
    plot_top_change_by_n(summary_by_method_n)

Running discrete comparison across retained methods.
Matrix bank: discrete_shared_matrices\ordinal_matrix_bank_seed_42_trials_5000.npz
Output folder: results_3_retained_methods
n values: [3, 4, 5, 6, 7, 8, 9]
alpha values: [2, 3, 4, 5, 6, 7, 8, 9]
baseline alpha: 2
ranking tolerance: 1e-10
log-weight bound: 8.0

Methods included:
- EVM
- RSM
- CSM
- HMM
- CMM

Running n=3; matrices=5000

Running n=4; matrices=5000

Running n=5; matrices=5000

Running n=6; matrices=5000

Running n=7; matrices=5000

Running n=8; matrices=5000

Running n=9; matrices=5000

Average by method:
  method  average_rank_reversal_frequency  average_top_change_frequency  \
0    CMM                         0.446514                      0.105829   
1    CSM                         0.555457                      0.148114   
2    EVM                         0.521629                      0.084057   
3    HMM                         0.226029                      0.011429   
4    RSM                         0.223371      

ValueError: This sheet is too large! Your sheet size is: 1400000, 11 Max sheet size is: 1048576, 16384

In [3]:
# =========================================================
# Recovery cell after Excel row-limit error
# Includes percentage labels above grouped bars
# =========================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.ticker import PercentFormatter


# Make sure the output folder exists
os.makedirs(OUTPUT_DIR, exist_ok=True)


# =========================================================
# Safe method order
# =========================================================

if "METHOD_ORDER" not in globals():
    METHOD_ORDER = ["EVM", "RSM", "CSM", "HMM", "CMM"]

if "PLOT_METHODS" not in globals():
    PLOT_METHODS = METHOD_ORDER

MARKERS = {
    "EVM": "o",
    "RSM": "s",
    "CSM": "^",
    "HMM": "D",
    "CMM": "x",
}

LINESTYLES = {
    "EVM": "-",
    "RSM": "--",
    "CSM": "-.",
    "HMM": ":",
    "CMM": "-",
}


# =========================================================
# Save helper
# =========================================================

def save_figure(fig, filename_base):
    jpeg_path = os.path.join(OUTPUT_DIR, f"{filename_base}.jpeg")
    pdf_path = os.path.join(OUTPUT_DIR, f"{filename_base}.pdf")

    fig.savefig(jpeg_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")

    print("Figure saved to:")
    print(jpeg_path)
    print(pdf_path)


# =========================================================
# Save outputs
# =========================================================

def save_outputs_recovery(
    results_df,
    matrix_level,
    summary_by_method_n,
    average_by_method,
    summary_by_method_n_alpha,
    summary_by_method_n_cr,
    failure_summary,
):
    """
    Save reviewer-ready outputs after the experiment has already run.

    The full alpha-level dataframe is not written to Excel because it exceeds
    Excel's maximum row limit. It is saved separately as a compressed CSV file.
    """

    excel_path = os.path.join(
        OUTPUT_DIR,
        "discrete_comparison_retained_methods_reviewer_ready.xlsx"
    )

    alpha_level_path = os.path.join(
        OUTPUT_DIR,
        "alpha_level_discrete_comparison_retained_methods.csv.gz"
    )

    latex_path = os.path.join(
        OUTPUT_DIR,
        "table_average_discrete_comparison_retained_methods.tex"
    )

    # Save the full alpha-level results separately
    results_df.to_csv(
        alpha_level_path,
        index=False,
        compression="gzip"
    )

    # Save only Excel-compatible summary sheets
    with pd.ExcelWriter(excel_path) as writer:

        average_by_method.to_excel(
            writer,
            sheet_name="average_by_method",
            index=False
        )

        summary_by_method_n.to_excel(
            writer,
            sheet_name="summary_method_n",
            index=False
        )

        summary_by_method_n_alpha.to_excel(
            writer,
            sheet_name="summary_method_n_alpha",
            index=False
        )

        summary_by_method_n_cr.to_excel(
            writer,
            sheet_name="summary_method_n_cr",
            index=False
        )

        matrix_level.to_excel(
            writer,
            sheet_name="matrix_level",
            index=False
        )

        failure_summary.to_excel(
            writer,
            sheet_name="failures",
            index=False
        )

    print("\nExcel summary file saved to:")
    print(excel_path)

    print("\nFull alpha-level results saved separately to:")
    print(alpha_level_path)

    # LaTeX table in percentages
    latex_df = average_by_method.copy()
    latex_df = latex_df[latex_df["method"].isin(PLOT_METHODS)]

    available_methods = [
        method for method in METHOD_ORDER
        if method in set(latex_df["method"])
    ]

    latex_df = latex_df.set_index("method").loc[available_methods].reset_index()

    latex_df["average_rank_reversal_frequency"] *= 100
    latex_df["average_top_change_frequency"] *= 100

    latex_df = latex_df.rename(columns={
        "method": "Method",
        "average_rank_reversal_frequency": "Average rank-reversal frequency (\\%)",
        "average_top_change_frequency": "Average top-change frequency (\\%)",
        "total_matrices": "Total matrices",
    })

    latex_table = latex_df.to_latex(
        index=False,
        float_format="%.1f",
        caption=(
            "Average discrete IOP rank-reversal frequency and top-ranked "
            "alternative change frequency across the retained priority "
            "derivation methods."
        ),
        label="tab:discrete_comparison_methods_revised",
        escape=False,
    )

    with open(latex_path, "w", encoding="utf-8") as f:
        f.write(latex_table)

    print("\nLaTeX table saved to:")
    print(latex_path)


# =========================================================
# Plot: grouped bars with exact percentage labels
# =========================================================

def plot_average_rank_and_top_grouped(average_by_method):
    plot_df = average_by_method[
        average_by_method["method"].isin(PLOT_METHODS)
    ].copy()

    available_methods = [
        method for method in METHOD_ORDER
        if method in set(plot_df["method"])
    ]

    plot_df = plot_df.set_index("method").loc[available_methods].reset_index()

    methods = plot_df["method"].to_numpy()
    x = np.arange(len(methods))
    width = 0.36

    rr = plot_df["average_rank_reversal_frequency"].to_numpy()
    top = plot_df["average_top_change_frequency"].to_numpy()

    fig, ax = plt.subplots(figsize=(9, 5))

    bars_rr = ax.bar(
        x - width / 2,
        rr,
        width,
        label="Rank reversal",
        hatch="//",
        edgecolor="black",
    )

    bars_top = ax.bar(
        x + width / 2,
        top,
        width,
        label="Top-ranked alternative change",
        hatch="\\\\",
        edgecolor="black",
    )

    # Exact percentage labels above bars
    ax.bar_label(
        bars_rr,
        labels=[f"{100 * value:.1f}%" for value in rr],
        padding=3,
        fontsize=9,
    )

    ax.bar_label(
        bars_top,
        labels=[f"{100 * value:.1f}%" for value in top],
        padding=3,
        fontsize=9,
    )

    max_value = max(rr.max(), top.max())
    upper_ylim = min(1.0, max(0.65, max_value + 0.08))

    ax.set_xlabel("Priority derivation method")
    ax.set_ylabel("Average frequency")
    ax.set_title("Discrete IOP rank reversal and top-ranked alternative changes")
    ax.set_xticks(x)
    ax.set_xticklabels(methods)
    ax.set_ylim(0, upper_ylim)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    ax.legend(frameon=False)

    plt.tight_layout()

    save_figure(
        fig,
        "discrete_bar_rank_reversal_top_change_retained_methods"
    )

    plt.close(fig)


# =========================================================
# Plot: rank reversal by matrix order
# =========================================================

def plot_rank_reversal_by_n(summary_by_method_n):
    plot_df = summary_by_method_n[
        summary_by_method_n["method"].isin(PLOT_METHODS)
    ].copy()

    available_methods = [
        method for method in METHOD_ORDER
        if method in set(plot_df["method"])
    ]

    n_values = sorted(plot_df["n"].unique())

    fig, ax = plt.subplots(figsize=(9, 5))

    for method in available_methods:

        df_m = plot_df[plot_df["method"] == method]

        ax.plot(
            df_m["n"],
            df_m["rank_reversal_frequency"],
            marker=MARKERS.get(method, "o"),
            linestyle=LINESTYLES.get(method, "-"),
            label=method
        )

    ax.set_xlabel(r"Matrix order $n$")
    ax.set_ylabel("IOP rank-reversal frequency")
    ax.set_title("Discrete IOP rank reversal by matrix order")
    ax.set_xticks(n_values)
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    ax.legend(frameon=False)

    plt.tight_layout()

    save_figure(
        fig,
        "discrete_rank_reversal_by_n_retained_methods"
    )

    plt.close(fig)


# =========================================================
# Plot: top-ranked alternative changes by matrix order
# =========================================================

def plot_top_change_by_n(summary_by_method_n):
    plot_df = summary_by_method_n[
        summary_by_method_n["method"].isin(PLOT_METHODS)
    ].copy()

    available_methods = [
        method for method in METHOD_ORDER
        if method in set(plot_df["method"])
    ]

    n_values = sorted(plot_df["n"].unique())

    fig, ax = plt.subplots(figsize=(9, 5))

    for method in available_methods:

        df_m = plot_df[plot_df["method"] == method]

        ax.plot(
            df_m["n"],
            df_m["top_change_frequency"],
            marker=MARKERS.get(method, "o"),
            linestyle=LINESTYLES.get(method, "-"),
            label=method
        )

    ax.set_xlabel(r"Matrix order $n$")
    ax.set_ylabel("Top-ranked alternative change frequency")
    ax.set_title("Top-ranked alternative changes by matrix order")
    ax.set_xticks(n_values)
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    ax.legend(frameon=False)

    plt.tight_layout()

    save_figure(
        fig,
        "discrete_top_change_by_n_retained_methods"
    )

    plt.close(fig)


# =========================================================
# Execute recovery
# =========================================================

save_outputs_recovery(
    results_df,
    matrix_level,
    summary_by_method_n,
    average_by_method,
    summary_by_method_n_alpha,
    summary_by_method_n_cr,
    failure_summary,
)

plot_average_rank_and_top_grouped(average_by_method)
plot_rank_reversal_by_n(summary_by_method_n)
plot_top_change_by_n(summary_by_method_n)

print("\nRecovery completed successfully.")


Excel summary file saved to:
results_3_retained_methods\discrete_comparison_retained_methods_reviewer_ready.xlsx

Full alpha-level results saved separately to:
results_3_retained_methods\alpha_level_discrete_comparison_retained_methods.csv.gz

LaTeX table saved to:
results_3_retained_methods\table_average_discrete_comparison_retained_methods.tex
Figure saved to:
results_3_retained_methods\discrete_bar_rank_reversal_top_change_retained_methods.jpeg
results_3_retained_methods\discrete_bar_rank_reversal_top_change_retained_methods.pdf
Figure saved to:
results_3_retained_methods\discrete_rank_reversal_by_n_retained_methods.jpeg
results_3_retained_methods\discrete_rank_reversal_by_n_retained_methods.pdf
Figure saved to:
results_3_retained_methods\discrete_top_change_by_n_retained_methods.jpeg
results_3_retained_methods\discrete_top_change_by_n_retained_methods.pdf

Recovery completed successfully.


In [4]:
# =========================================================
# Reviewer Table: Discrete percentages by n and method
# Comment 2: Figures 3 and 5 should be complemented with
# method-by-order percentages.
# =========================================================

import os
import pandas as pd

# ---------------------------------------------------------
# Safety checks
# ---------------------------------------------------------

if "summary_by_method_n" not in globals():
    raise NameError(
        "summary_by_method_n is not defined. "
        "Run the discrete comparison experiment first."
    )

if "METHOD_ORDER" not in globals():
    METHOD_ORDER = ["EVM", "RSM", "CSM", "HMM", "CMM"]

if "N_VALUES" not in globals():
    N_VALUES = list(range(3, 10))

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = "results_3_retained_methods"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ---------------------------------------------------------
# Build percentage tables
# ---------------------------------------------------------

df = summary_by_method_n.copy()

df = df[df["method"].isin(METHOD_ORDER)]
df = df[df["n"].isin(N_VALUES)]

# Convert frequencies to percentages
df["rank_reversal_percent"] = 100 * df["rank_reversal_frequency"]
df["top_change_percent"] = 100 * df["top_change_frequency"]

rank_table = (
    df.pivot(index="n", columns="method", values="rank_reversal_percent")
    .reindex(index=N_VALUES, columns=METHOD_ORDER)
    .round(1)
)

top_table = (
    df.pivot(index="n", columns="method", values="top_change_percent")
    .reindex(index=N_VALUES, columns=METHOD_ORDER)
    .round(1)
)

# Add n as a visible column
rank_table_out = rank_table.reset_index()
top_table_out = top_table.reset_index()


# ---------------------------------------------------------
# Save Excel version
# ---------------------------------------------------------

excel_path = os.path.join(
    OUTPUT_DIR,
    "table_discrete_percentages_by_n_and_method.xlsx"
)

with pd.ExcelWriter(excel_path) as writer:
    rank_table_out.to_excel(
        writer,
        sheet_name="rank_reversal",
        index=False
    )
    top_table_out.to_excel(
        writer,
        sheet_name="top_change",
        index=False
    )


# ---------------------------------------------------------
# Build LaTeX table with two panels
# ---------------------------------------------------------

def latex_panel(table, panel_title):
    lines = []
    lines.append(r"\multicolumn{6}{l}{\textit{" + panel_title + r"}} \\")
    lines.append(r"\toprule")
    lines.append(r"$n$ & EVM & RSM & CSM & HMM & CMM \\")
    lines.append(r"\midrule")

    for _, row in table.iterrows():
        line = (
            f"{int(row['n'])} & "
            f"{row['EVM']:.1f} & "
            f"{row['RSM']:.1f} & "
            f"{row['CSM']:.1f} & "
            f"{row['HMM']:.1f} & "
            f"{row['CMM']:.1f} \\\\"
        )
        lines.append(line)

    return "\n".join(lines)


latex_lines = []

latex_lines.append(r"\begin{table}[htbp]")
latex_lines.append(r"\centering")
latex_lines.append(r"\caption{Discrete intensity-of-preference rank-reversal and top-ranked alternative change percentages by matrix order and priority derivation method. Percentages are computed over the same fixed set of 5000 ordinal matrices for each matrix order \(n\), comparing the baseline \(A(2)\) with the uniformly intensified matrices \(A(\alpha)\), \(\alpha\in\{3,\ldots,9\}\).}")
latex_lines.append(r"\label{tab:discrete_percentages_by_n_method}")
latex_lines.append(r"\begin{tabular}{rrrrrr}")
latex_lines.append(latex_panel(rank_table_out, "Panel A: IOP rank reversal percentages"))
latex_lines.append(r"\midrule")
latex_lines.append(latex_panel(top_table_out, "Panel B: Top-ranked alternative change percentages"))
latex_lines.append(r"\bottomrule")
latex_lines.append(r"\end{tabular}")
latex_lines.append(r"\end{table}")

latex_table = "\n".join(latex_lines)

latex_path = os.path.join(
    OUTPUT_DIR,
    "table_discrete_percentages_by_n_and_method.tex"
)

with open(latex_path, "w", encoding="utf-8") as f:
    f.write(latex_table)


# ---------------------------------------------------------
# Print and display
# ---------------------------------------------------------

print("Panel A: IOP rank reversal percentages")
display(rank_table_out)

print("\nPanel B: Top-ranked alternative change percentages")
display(top_table_out)

print("\nFiles saved:")
print(excel_path)
print(latex_path)

print("\nLaTeX table:")
print(latex_table)

Panel A: IOP rank reversal percentages


method,n,EVM,RSM,CSM,HMM,CMM
0,3,0.0,0.0,0.0,0.0,14.8
1,4,5.0,0.0,14.7,0.0,46.2
2,5,36.3,6.2,41.0,5.8,40.6
3,6,60.9,15.8,64.2,16.7,44.9
4,7,79.6,30.9,81.5,30.5,49.3
5,8,88.9,45.4,91.5,45.2,54.5
6,9,94.5,58.0,95.9,60.0,62.2



Panel B: Top-ranked alternative change percentages


method,n,EVM,RSM,CSM,HMM,CMM
0,3,0.0,0.0,0.0,0.0,7.2
1,4,0.0,0.0,6.6,0.0,14.0
2,5,6.8,0.0,10.4,0.0,11.6
3,6,10.6,0.0,15.4,0.6,10.8
4,7,12.1,0.2,20.3,1.1,10.8
5,8,13.9,0.6,25.4,2.6,10.1
6,9,15.5,0.9,25.6,3.8,9.7



Files saved:
results_3_retained_methods\table_discrete_percentages_by_n_and_method.xlsx
results_3_retained_methods\table_discrete_percentages_by_n_and_method.tex

LaTeX table:
\begin{table}[htbp]
\centering
\caption{Discrete intensity-of-preference rank-reversal and top-ranked alternative change percentages by matrix order and priority derivation method. Percentages are computed over the same fixed set of 5000 ordinal matrices for each matrix order \(n\), comparing the baseline \(A(2)\) with the uniformly intensified matrices \(A(\alpha)\), \(\alpha\in\{3,\ldots,9\}\).}
\label{tab:discrete_percentages_by_n_method}
\begin{tabular}{rrrrrr}
\multicolumn{6}{l}{\textit{Panel A: IOP rank reversal percentages}} \\
\toprule
$n$ & EVM & RSM & CSM & HMM & CMM \\
\midrule
3 & 0.0 & 0.0 & 0.0 & 0.0 & 14.8 \\
4 & 5.0 & 0.0 & 14.7 & 0.0 & 46.2 \\
5 & 36.3 & 6.2 & 41.0 & 5.8 & 40.6 \\
6 & 60.9 & 15.8 & 64.2 & 16.7 & 44.9 \\
7 & 79.6 & 30.9 & 81.5 & 30.5 & 49.3 \\
8 & 88.9 & 45.4 & 91.5 & 45.2 & 54.5